# 8. Analyse critique et pistes d'amélioration

Ce notebook synthétise les **éléments réellement mesurés** dans le projet, permet d'interpréter les **résultats du jeu de test** (une fois `trainYolov9.ipynb` et `evalYolov9.ipynb` exécutés sur GPU), analyse les **limites du modèle** et propose des **pistes d'amélioration justifiées**.

In [ ]:
from pathlib import Path
from collections import Counter
import yaml
import pandas as pd

DATA_DIR = Path("../trafic_data").resolve()
YOLO_DIR = Path("../yolov9").resolve()
RESULTS_CSV = YOLO_DIR / "runs/train/yolov9c_vehicles_e10/results.csv"
BEST_PT = YOLO_DIR / "runs/train/yolov9c_vehicles_e10/weights/best.pt"

with open(DATA_DIR / "data_1.yaml", "r", encoding="utf-8") as f:
    names = yaml.safe_load(f)["names"]

summary = []
cls_counts = {}
for split in ["train", "valid", "test"]:
    imgs = list((DATA_DIR / split / "images").glob("*"))
    c = Counter()
    for lbl in (DATA_DIR / split / "labels").glob("*.txt"):
        for line in lbl.read_text(encoding="utf-8").splitlines():
            if line.strip():
                c[int(line.split()[0])] += 1
    cls_counts[split] = c
    summary.append({
        "split": split,
        "images": len(imgs),
        "boites": sum(c.values()),
        "classes_presentes": f"{len(c)}/{len(names)}",
    })

print("=== Faits mesurés sur le dataset nettoyé ===")
print(pd.DataFrame(summary).to_string(index=False))

if RESULTS_CSV.exists():
    df_res = pd.read_csv(RESULTS_CSV)
    df_res.columns = [c.strip() for c in df_res.columns]
    print("\n=== Dernières métriques mesurées (validation pendant entraînement) ===")
    cols = [c for c in ["epoch", "metrics/precision", "metrics/recall", "metrics/mAP_0.5", "metrics/mAP_0.5:0.95"] if c in df_res.columns]
    print(df_res[cols].tail(3).to_string(index=False))
else:
    print(f"\n[En attente d'entraînement GPU] Aucun résultat mesuré pour l'instant ({RESULTS_CSV} absent, best.pt={BEST_PT.exists()}).")

## 1. Interprétation des résultats et comparaison des catégories de véhicules

*(Les métriques quantitatives de test s'afficheront dans `evalYolov9.ipynb` après exécution sur GPU ; les tendances ci-dessous s'appuient sur la structure mesurée du dataset.)*

- **Classes attendues en haut du classement (mAP@50 élevé) :**
  - `car` (`3 848` train / `784` test), `rickshaw` (`2 836` / `500`), `bus` (`2 465` / `444`), `three wheelers -CNG-` (`2 281` / `453`) et `motorbike` (`1 660` / `289`).
  - **Pourquoi :** fort volume d'exemples variés, silhouettes distinctives (`bus`, `rickshaw`, `CNG`) et forte présence dans les poids préentraînés COCO (`car`, `bus`, `motorbike`).
- **Classes attendues plus faibles ou instables :**
  - **Classes rares :** `garbagevan` (`3` train / `0` valid / `0` test — non évaluable sur `valid` et `test`), `policecar` (`23` / `8`), `scooter` (`30` / `7`), `army vehicle` (`35` / `8`), `taxi` (`36` / `5`), `ambulance` (`54` / `16`), `minibus` (`77` / `16`).
  - **Classes visuellement proches :** `car` vs `suv` vs `taxi` ; `van` vs `minivan` vs `minibus` ; `motorbike` vs `scooter` ; `pickup` vs `truck`. Les confusions réduisent directement la `Precision` et le `Recall` par classe.

## 2. Limites identifiées

1. **Distribution et déséquilibre des données :**
   - Ratio global `max/min` de **`1824,7x`** entre `car` (`5 474` boîtes) et `garbagevan` (`3` boîtes).
   - `5` classes absentes de `valid` (`auto rickshaw`, `human hauler`, `ambulance`, `army vehicle`, `garbagevan`) : le modèle `best.pt` sélectionné sur `valid` n'est donc pas optimisé sur ces 5 classes.
   - Présence de `44` paires d'images très similaires (`dHash <= 4` : `34` train/valid, `8` train/test, `2` valid/test) pouvant légèrement surestimer les scores si des scènes quasi identiques apparaissent dans deux splits.
2. **Nombre d'époques limité (`10` époques) :**
   - `10` époques suffisent pour adapter rapidement les classes fréquentes déjà proches de COCO, mais restent courtes pour stabiliser les classes rares (`< 100` exemples) et les têtes de détection `DualDDetect` sur `21` classes.
3. **Qualité et difficulté des annotations :**
   - Des annotations invalides existaient dans le jeu brut (`2` fichiers vides en `train` et `24` lignes `0 0 0 0` dans `valid`, nettoyées à l'étape 2).
   - Scènes routières denses avec nombreux petits véhicules éloignés et occultations partielles, ce qui pénalise le `Recall` et la métrique stricte `mAP@50-95`.

## 3. Pistes d'amélioration proposées (à tester après la baseline)

1. **Prolonger l'entraînement (30 à 50 époques) avec `--cos-lr` :**
   - *Pertinence :* `hyp.scratch-high.yaml` utilise un *warmup* de 3 époques ; sur 10 époques, le modèle passe 30 % du temps en échauffement. Monter à 30–50 époques avec décroissance cosinus permet une meilleure convergence sur les 21 classes.
2. **Rééquilibrage des classes rares (`--image-weights` / découpage stratifié / fusion ou collecte) :**
   - *Pertinence :* `--image-weights` dans `train_dual.py` sur-échantillonne les images contenant des classes sous-représentées. Un découpage stratifié garantirait aussi la présence des 21 classes dans `valid` et `test` (ou le regroupement des classes à effectif quasi nul comme `garbagevan`).
3. **Déduplication perceptuelle inter-splits et ajustement d'inférence :**
   - *Pertinence :* regrouper les images quasi-identiques (`dHash <= 4`) dans le même split évite toute fuite de données, et tester `--img 800` ou l'ajustement du `--conf-thres` selon le cas d'usage améliore la détection des petits véhicules lointains.

## 4. Distinction entre résultats réellement mesurés et améliorations à tester

| Élément | Statut | Détail observé |
| :--- | :--- | :--- |
| Effectifs après nettoyage (`train` / `valid` / `test`) | **Réellement mesuré** | `2 297` / `298` / `405` images (`18 326` / `2 560` / `3 438` boîtes) |
| Nettoyage des annotations invalides | **Réellement mesuré** | `2` labels vides (`train`) et `24` lignes `0 0 0 0` (`valid`) supprimés |
| Déséquilibre des classes & couverture des splits | **Réellement mesuré** | Ratio `1824,7x`, `5` classes absentes de `valid`, `1` absente de `test` |
| Doublons inter-splits | **Réellement mesuré** | `0` doublon exact SHA-256 ; `44` paires similaires `dHash <= 4` |
| Transfer Learning `yolov9-c.pt` (`nc=21`) & coût CPU | **Réellement mesuré** | `1 448/1 460` tenseurs transférés ; `38,00 s/batch` sur CPU local |
| Scores `Precision`, `Recall`, `mAP@50`, `mAP@50-95` sur `test` | **À mesurer (après run GPU)** | Dépend de l'exécution de `trainYolov9.ipynb` puis `evalYolov9.ipynb` |
| Gain avec 30–50 époques, `--image-weights`, stratification | **À tester** | Pistes proposées, non encore entraînées ni comparées à la baseline |